In [1]:
import pandas as pd
import glob
import json
import os
DATA_DIR = "data/"

In [ ]:
def build_dpo_train_dataset(unsafe_only=True):
    print("Building DPO JSONL Training Dataset...\n")
    
    # 1. Load the rich persona descriptions
    try:
        with open('personas_desc.json', 'r', encoding='utf-8') as f:
            persona_descriptions = json.load(f)
        print("Successfully loaded rich system prompts from personas_desc.json")
    except Exception as e:
        print(f"Error loading personas_desc.json: {e}")
        return

    # 2. Load BOTH empty/baseline datasets to use as our "Rejected" responses
    baseline_dict = {}
    for baseline_file in ['empty_unsafe_dataset.csv', 'empty_safe_dataset.csv']:
        filepath = os.path.join(DATA_DIR, baseline_file)
        if os.path.exists(filepath):
            try:
                baseline_df = pd.read_csv(filepath, encoding='utf-8')
                col_name = 'rejected_response' if 'rejected_response' in baseline_df.columns else 'preferred_response'
                baseline_dict.update(dict(zip(baseline_df['prompt'], baseline_df[col_name])))
                print(f"Loaded {len(baseline_df)} baseline responses from {baseline_file}.")
            except Exception as e:
                print(f"Error loading {baseline_file}: {e}")

    print(f"Total baseline responses loaded: {len(baseline_dict)}\n")

    # 3. Find all persona datasets
    persona_files = [f for f in glob.glob(os.path.join(DATA_DIR, '*_dataset.csv')) if 'empty' not in f]
    
    if unsafe_only:
        output_file = os.path.join('dpo_training_unsafe_data.jsonl')
    else:
        output_file = os.path.join('dpo_training_data.jsonl')
    total_pairs = 0
    missing_baselines = 0
    skipped_test_rows = 0

    # 4. Open the JSONL file for writing
    with open(output_file, 'w', encoding='utf-8') as outfile:
        
        for file in persona_files:
            try:
                df = pd.read_csv(file, encoding='utf-8')
                
                # Check if the split column exists to avoid errors
                if 'split' not in df.columns:
                    print(f"Warning: No 'split' column found in {file}. Skipping file.")
                    continue

                filename = os.path.basename(file)
                
                # Extract persona name
                if '_safe_dataset.csv' in filename:
                    if unsafe_only: 
                        continue
                    persona_name = filename.replace('_safe_dataset.csv', '').replace('_', ' ')
                elif '_unsafe_dataset.csv' in filename:
                    persona_name = filename.replace('_unsafe_dataset.csv', '').replace('_', ' ')
                else:
                    continue 
                
                # Fetch raw description
                persona_desc = persona_descriptions.get(persona_name, "")
                
                # Match your exact requested format
                full_system_prompt = f"You are a {persona_name}. {persona_desc}".strip()
                
                for index, row in df.iterrows():
                    if str(row['split']).strip().lower() != 'train':
                        skipped_test_rows += 1
                        continue

                    prompt = str(row['prompt'])
                    chosen_response = str(row['preferred_response']).strip()
                    rejected_response = baseline_dict.get(prompt)
                    
                    if not rejected_response or pd.isna(rejected_response):
                        print(f"Warning: No baseline response found for prompt: '{prompt}' in file: {file}. Skipping this row.")
                        missing_baselines += 1
                        continue 
                    
                    # 5. Construct the DPO JSON object
                    dpo_example = {
                        "system": full_system_prompt,
                        "prompt": prompt,
                        "chosen": chosen_response,
                        "rejected": str(rejected_response).strip()
                    }
                    
                    # Write to the JSONL file
                    json.dump(dpo_example, outfile, ensure_ascii=False)
                    outfile.write('\n')
                    total_pairs += 1
                    
            except Exception as e:
                print(f"Error processing {file}: {e}")

    print("\n" + "="*50)
    print("PHASE 5 COMPLETE: DATASET READY FOR FINE-TUNING")
    print("="*50)
    print(f"Total DPO pairs generated (Train Only): {total_pairs}")
    print(f"Rows held back for Test Set: {skipped_test_rows}")
    print(f"Your final, highly detailed training file is saved as: {output_file}")